In [ ]:
from IPython.display import Image, HTML, display
import warnings
image_path = "Logos.jpeg"

img_html = Image(url=image_path, width=800)._repr_html_()

centered_image_html = f"<div style='text-align:center;'>{img_html}</div>"

display(HTML(centered_image_html))

warnings.filterwarnings('ignore')

<div style="text-align:center"; font-size: 32px;>
    
# Geospatial data analysis

</div>

**Author:** Naziru Halilu


# Class 1: Vector data

<span style="font-size:120%">
This class aims to introduce students to the manipulation of vector data using programming code.
The following data will be used in the following activity:
    
- Agricultural plots (Common Agricultural Policy information)
- Soil map (https://geoportal.navarra.es/es/idena/descargar) ['EDAFOL_Pol_Suelos25m']
- Main rivers (https://geoportal.navarra.es/es/idena/descargar) ['HIDROG_Pol_RioPrincipal']

In this class, we will seek to obtain the percentage of soil in different plots. A single plot will be selected from the municipality of ‘Miranda de Arga’, as well as those located within a distance of 500 m. Finally, the distance to rivers will be calculated from the selected plots, and information will be extracted about the closest one.

This activity seeks to enable students to:

- Visualize vector information
- Manipulate vector data (intersect, dissolve, buffer, extract information, and perform distance analysis)
- Export the results obtained
</span>

## Import libraries

In [ ]:
# Manipulating directories and paths
import os #More information https://docs.python.org/3/library/os.html

# Data manipulation and analysis.
import pandas as pd 
import numpy as np

# Spatial data manipulation
import geopandas as gpd #More information https://geopandas.org/en/stable/docs.html

# Graphics design
import matplotlib.pyplot as plt # More information https://matplotlib.org/

In [ ]:
pip install descartes

# Load vector data

<span style="font-size:120%">
To visualize the data and understand its contents, you can use commands similar to those used to manipulate a data frame, such as .columns or writing the variable name. However, the presence of a geometry attribute will allow us to locate each of the elements of our geodataframe in space. In addition to spatial visualization, the presence of a geometry will allow us to extract information from overlapping areas, determine the distances between objects, and many other features.

</span>

In [ ]:
path=os.getcwd()

### Agricultural plots

In [ ]:
# Set the address of the directory we are working in
os.chdir(path+'\\Data\\')

In [ ]:
# Read the vector file
plots=gpd.read_file('PAC\\PAC_2017.shp')

In [ ]:
# Visualize the name of the columns
plots.columns

In [ ]:
# Show the geodataframe
plots

<span style="font-size:120%">
Various libraries can be used to visualize a geodataframe. Within GeoPandas, the .plot() command allows for quick and easy visualization of geometries. However, for greater versatility, libraries such as matplotlib and seaborn are the most widely implemented.

For more information on these libraries and graph types, please visit the following pages:

- Matplotlib: https://matplotlib.org/stable/
- Seaborn: https://seaborn.pydata.org/

</span>

In [ ]:
# Create a new figure and a set of axes for the plot 
fig, ax = plt.subplots(figsize=(10, 10))

# Indicate which column to represent
plots.plot(column='ClasGen', legend=True, ax=ax,legend_kwds={
                  'loc': 'center left',  
                  'bbox_to_anchor': (1, 0.5), 
                  'title': 'Crops and land use' 
              })



plt.show()

<span style="font-size:120%">
In addition to visualizing the location of different parcels using geometry, this attribute allows us to obtain new information such as area in the case of polygons, and perimeters in the case of lines and polygons.
It is very important for calculating areas and distances that the reference system be in projected coordinates, since the result of these operations will be m2 or m, respectively.
To consult the reference system of a geodataframe, it is necessary to use the .crs command. If the reference system is not the desired one, it can be transformed using, for example, .to_crs({'init':'epsg:25830'})
</span>

In [ ]:
print('The reference system is: ',plots.crs)

In [ ]:
# Calculate the area of each agricultural plot
plots['GEOM_AREA_PLOT']=plots['geometry'].area

### Soil map

In [ ]:
# Read the soil file
soil=gpd.read_file('EDAFOL_Pol_Suelos25m\\EDAFOL_Pol_Suelos25m.shp')

In [ ]:
# Visualize the data
soil.columns

In [ ]:
soil.plot()

# Data manipulation

## Intersection

<span style="font-size:120%">
We now have two layers loaded: the agricultural plots and the soil map. We want to know the soil type of each agricultural plot. To do this, we must overlap the layers by means of an intersection, extracting the attributes of both layers. As with the area calculation, it is necessary to verify the reference systems; it is essential that they are identical.
</span>

In [ ]:
# Check if both reference systems are equal
print(plots.crs==soil.crs)

In [ ]:
# To avoid having a large number of columns, we can select only those that interest us.
soil=soil[['SOILTAXON1','CLASIF_SC1','MUNICIPIO', 'geometry']]
soil.columns=['SOILTAXON1','CLASIF_SC1','NAME_MUNIC', 'geometry']

<span style="font-size:120%">
To intersect two polygons, use the .overlay command. You must specify the two layers to be intersected, as well as the operation you want to perform. In this case, we are looking for an intersection.
Using this command, you can perform other operations between geometries, such as union, difference, and others. You can see examples and find more information at the following link:

https://geopandas.org/en/stable/docs/reference/api/geopandas.overlay.html
</span>

In [ ]:
intersection=gpd.overlay(plots,soil,how='intersection')

In [ ]:
# Since there are areas without soil map information, some rows are NaN. To filter out this data, we can apply '.dropna()'.
intersection=intersection.dropna()

In [ ]:
# Create a new figure and a set of axes for the plot 
fig, ax = plt.subplots(figsize=(10, 10))

# # Get the bounds of the intersection GeoDataFrame
# x_min, y_min, x_max, y_max = intersection.total_bounds

# Indicate which column to represent
intersection.plot(column='ClasGen', legend=True,ax=ax,             legend_kwds={
                  'loc': 'center left',  
                  'bbox_to_anchor': (1, 0.5), 
                  'title': 'Type of soil' 
              })


# Set the zoom zone
ax.set_xlim(600900, 602000)
ax.set_ylim(4702000, 4703000)

plt.show()

<span style="font-size:120%">
We have completed the intersection. In this part of the exercise, we will filter the geodataframe to select the municipality of 'Miranda de Arga' and see the result of the intersection, focusing on a single plot.
</span>

In [ ]:
# As in a dataframe, we can see the number of geometries that we have in each municipality.
intersection.groupby('NAME_MUNIC')['NAME_MUNIC'].count().sort_values(ascending=False)

In [ ]:
# Filter using .loc
intersection=intersection.loc[intersection.NAME_MUNIC=='Miranda de Arga']

In [ ]:
# Quick visualization using .plot() of geopandas
intersection.plot()

<span style="font-size:120%">
After intersection, each original agricultural plot has as many geometries as different geometries of soil had been intersected. To be conscious about these, we can group them by considering an ID of each agricultural plot and see how many types of soil and geometries each agricultural plot has. 
</span>

In [ ]:
# Create an ID for each agricultural plot
intersection['IDPLOT']=intersection['POLIGONO'].astype(str)+intersection['PARCELA'].astype(str)+intersection['RECINTO'].astype(str)
       

In [ ]:
# Know hoy many geometries each agricultural plot has.
intersection.groupby('IDPLOT')['IDPLOT'].count().sort_values(ascending=False)

In [ ]:
# Know how many type of soil each agricultural plot has.
# Filter to get one agricultural plot
example=intersection.loc[intersection.IDPLOT=='1316.01.0']
# The calculate how many type of soil each agricultural plot has.
print(list(set(example.SOILTAXON1)))

In [ ]:
# Plot the type of soils
fig, ax = plt.subplots(figsize=(10, 10))

example.plot(column='SOILTAXON1', legend=True, ax=ax,  legend_kwds={
                  'loc': 'center left',  
                  'bbox_to_anchor': (1, 0.5), 
                  'title': 'Type of soil' 
              })
plt.show()

In [ ]:
example

## Dissolve

<span style="font-size:120%">
We are looking to determine the quantity of each soil type present in each agricultural plot. Therefore, all soils belonging to the same SOILTAXON1 must be grouped together. To do this, the dissolve method can be implemented. <br>
<br>

__.dissolve()__ combines the geometries of multiple rows in a GeoDataFrame into a single geometry, based on the values in one or more columns. This is similar to the SQL GROUP BY operation, but for geographic data.

To carry out this method, it is necessary to indicate which attributes must be implemented to dissolve the geometry. In this case, we want all identical soils within each plot to appear only once. Therefore, the attributes by which the geometry must be dissolved are: the plot indicator 'IDPLOT' and the soil type 'SOILTAXON1'.


- What would happen if we dissolve taking into account only 'SOILTAXON1'?
</span>

In [ ]:
# Apply the dissolve method
dissolved=intersection.dissolve(by=['IDPLOT','SOILTAXON1'], as_index=False)
len(dissolved)

In [ ]:
# See if the number of geometries per plot decreased
dissolved.groupby('IDPLOT')['IDPLOT'].count().sort_values(ascending=False)

In [ ]:
# Re-plot the example plot. Any differences?
example=dissolved.loc[dissolved.IDPLOT=='1316.01.0']

fig, ax = plt.subplots(figsize=(10, 10))
example.plot(column='SOILTAXON1', legend=True, figsize=(10, 10), ax=ax,             legend_kwds={
                  'loc': 'center left',  
                  'bbox_to_anchor': (1, 0.5), 
                  'title': 'Type of soil' 
              })

plt.show()

<span style="font-size:120%">
We will select the previously selected plot. We now want to identify plots located within 500 m of each other. To do this, we will generate a 500 m buffer zone, taking into account the geometry of our chosen plot. As with any spatial operation, it is vital to know the reference system. In this case, since we are using a projected reference system, we know that the surrounding buffer will be expressed in meters.
</span>

In [ ]:
# Apply the dissolve method
examplebuffer=example.dissolve(by=['IDPLOT'], as_index=False)
buffer=examplebuffer[:]
buffer.geometry=buffer.buffer(500)


In [ ]:
buffer

In [ ]:
# Create a Matplotlib Axes object
# This will be the container for your plot
fig, ax = plt.subplots(figsize=(10, 10))

# 2. Plot the first layer (the buffered layer)
# Pass the 'ax' object to the plot method
buffer.plot(ax=ax, color='blue', alpha=0.1, label='Buffer (500 m)')

# 3. Plot the second layer (the original layer)
# Plot it on the same 'ax' object and style it differently
example.plot(ax=ax, color='red')

# Optional: Add a legend, title, and show the plot
ax.set_title('Plot picked and 500 m Buffer')

plt.show()

In [ ]:
buffer=buffer[['geometry']]

In [ ]:
# Apply an intersection between the buffer layer and the agricultural plot dissolved
SecondIntersection=gpd.overlay(dissolved,buffer,how='intersection')
SecondIntersection

In [ ]:
IDs=list(set(SecondIntersection.IDPLOT))
# IDs

In [ ]:
len(IDs)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

# Plot first layer
buffer.plot(ax=ax, color='blue', alpha=0.1, label='Buffer (500 m)')
# Plot secod layer
SecondIntersection.plot(column='IDPLOT',color='black',edgecolor='black',linewidth=0.5, legend=False,ax=ax)
# Plot third layer
example.plot(color='yellow',ax=ax)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

# Plot first layer
intersection.plot(ax=ax, color='blue', alpha=0.1)

# Plot the second layer (the original layer)
# Plot it on the same 'ax' object and style it differently
example.plot(ax=ax, color='red')

plt.show()

In [ ]:
# Select those plots within the buffer 
selected= dissolved[dissolved['IDPLOT'].isin(IDs)]
selected

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

# Plot first layer
buffer.plot(ax=ax, color='blue', alpha=0.1, label='Buffer (500 m)')
# Plot second layer
selected.plot(column='IDPLOT',ax=ax, legend=False, figsize=(10, 10),legend_kwds={
                  'loc': 'center left',  
                  'bbox_to_anchor': (1, 0.5), 
                  'title': 'Type of soil' 
              })


plt.show()

In [ ]:
selected=selected.copy()

In [ ]:
# Calculate the area of each geometry
selected['AREA_SOIL']=selected['geometry'].area
# selected

In [ ]:
# Calculate the percentage of each soil's geometry
selected['PERC_SOIL']=((selected['AREA_SOIL']/selected['GEOM_AREA_PLOT'])*100).round(2)
# selected

In [ ]:
# Plot
fig, ax = plt.subplots(figsize=(15, 15))

selected.plot(column='SOILTAXON1', ax=ax, legend=True,
              legend_kwds={
                  'loc': 'center left',  
                  'bbox_to_anchor': (1, 0.5), 
                  'title': 'Type of soil' 
              })

plt.tight_layout()
plt.show()

In [ ]:
example=selected.loc[selected.IDPLOT=='1316.01.0']
fig, ax = plt.subplots(figsize=(10, 10))

# Plot the example GeoDataFrame, using the SOILTAXON1 column for the colors
example.plot(column='SOILTAXON1',
             ax=ax,
             legend=True,
             edgecolor='black',
             linewidth=0.5,
             legend_kwds={
                  'loc': 'center left',  
                  'bbox_to_anchor': (1, 0.5), 
                  'title': 'Type of soil' 
              })
            

# Get the legend that GeoPandas just created
legend = ax.get_legend()

# Create a dictionary to map each soil type to its percentage
# This assumes that each row in 'example' already contains the correct percentage for that taxon.
# .drop_duplicates() is used to return a single percentage per taxon if there are multiple rows.

taxon_percentages = example.drop_duplicates(subset=['SOILTAXON1']) \
                           .set_index('SOILTAXON1')['PERC_SOIL'] \
                           .round(1) \
                           .to_dict()

# Modify legend labels to add percentage
for text in legend.get_texts():
    original_label = text.get_text()
    if original_label in taxon_percentages:
        # Crea la nueva etiqueta con el porcentaje y el tipo de suelo
        new_label = f"{original_label} ({taxon_percentages[original_label]}%)"
        text.set_text(new_label)

plt.tight_layout()
plt.show()

In [ ]:
example

## Distance calculation

In [ ]:
# Load the river file
river=gpd.read_file('HIDROG_Pol_RioPrincipal\\HIDROG_Pol_RioPrincipal.shp')
river.plot(color='blue')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

# Plot first layer
river.plot(ax=ax, color='blue', alpha=1, label='Buffer (500 m)')
# Plot second layer
selected.plot(column='ClasGen',ax=ax, legend=True, figsize=(10, 10),legend_kwds={
                  'loc': 'center left',  
                  'bbox_to_anchor': (1, 0.5), 
                  'title': 'Type of soil' 
              })

x_min, y_min, x_max, y_max = selected.total_bounds

# `ax.set_xlim` y `ax.set_ylim` ajustan el zoom del gráfico
ax.set_xlim(x_min-10000, x_max+10000)
ax.set_ylim(y_min-10000, y_max+10000)

plt.show()

In [ ]:
print(river.crs==selected.crs)

In [ ]:
# Create two new columns for the distance and the river name
selected['Dist_river'] = np.nan
selected['Name_river'] = ""

# Iterate over each plot to find the minimum distance and the river
for idx, plot in selected.iterrows():
    # Calculate the distances from the plot to all river sections
    distances = river.geometry.distance(plot.geometry)
    
    # Find the index of the river section with the minimum distance
    min_dist_idx = distances.idxmin()
    
    # Use the index to get the minimum distance and the name of the river
    min_distance = distances.loc[min_dist_idx]
    closest_river_name = river.loc[min_dist_idx, 'NOMBRE'] 
    
    # Save the results in the new plot columns
    selected.loc[idx, 'Dist_river'] = min_distance
    selected.loc[idx, 'Name_river'] = closest_river_name

print(selected[['IDPLOT', 'Dist_river', 'Name_river']].head())

# Export result

In [ ]:
selected.to_file(path+'\\Results\\Plots_Clas1.shp')

In [ ]:
selected=selected[['IDPLOT', 'SOILTAXON1', 'geometry', 'IDCOMARCA', 'COMARCA', 'GEOM_AREA',
       'PROVINCIA', 'MUNICIPIO', 'POLIGONO', 'PARCELA', 'RECINTO', 'COEF_AUTO',
       'COEF_REGAD', 'PRD_CODIGO', 'PRD_DESC_1', 'IDClasGen', 'IDClasEsp',
       'ClasGen', 'ClasEsp', 'REFCAT', 'GEOM_AREA_PLOT', 'CLASIF_SC1',
       'NAME_MUNIC', 'AREA_SOIL', 'PERC_SOIL', 'Dist_river', 'Name_river']].astype(str)

In [ ]:
selected.to_csv(path+'\\Results\\Plots_Clas1.csv')